### 坏图检查

In [1]:
from pathlib import Path
from PIL import Image, ImageFile
import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False  # 扫描时保持严格，才能找出坏的

def is_ok(p: Path) -> bool:
    try:
        with Image.open(p) as im:
            im.verify()  # 快速校验文件结构
        # verify 后需要重新 open 才能 load
        with Image.open(p) as im:
            im.load()    # 真正解码一次，能抓到“truncated”
        return True
    except Exception:
        return False

# root = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a04_whole_raw_data/air2_0701_0823")
# root = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a04_whole_raw_data/southfarm2_0712_0823")
root = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data")
paths = [p for p in root.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png"}]

bad = []

for p in tqdm.tqdm(paths):
    if not is_ok(p):
        bad.append(p)

Path("bad_images.txt").write_text("\n".join(map(str, bad)))
print("bad =", len(bad))


100%|██████████| 32519/32519 [26:07<00:00, 20.75it/s]

bad = 0


In [ ]:
# 批量移动图片到delete文件夹
from os import mkdir

images_list = [

]

delete_folder = "/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a04_whole_raw_data/air2_0701_0823/delete"

# 创建delete文件夹
if not Path(delete_folder).exists():
    mkdir(delete_folder)

for image_path in images_list:
    image_name = image_path.split("/")[-1]
    new_path = f"{delete_folder}/{image_name}"
    Path(image_path).rename(new_path)


### 没有GT，直接开始预测Raw数据

In [2]:
from pathlib import Path
import site
import fiftyone as fo
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction


# =========================
# Config（极简版）
# =========================
DATA_ROOT = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data")

# VERSION = "sahi_null_run_whole_rawData_v6"
VERSION = "sahi_null_run_whole_rawData_rest7datasets_v7"

MODEL_DIR = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/04_swd_hbb/null_image_trained_final_checkpoint")

MODEL_NAMES = [
    # "yolo11x_20pct_null_images_add_rawData_batch_8_final.pt",
    # "yolo11m_20pct_null_images_add_rawData_batch_16_final.pt",
    # "yolo11m_20pct_null_images_add_rawData_batch_8_final.pt",
    "yolo11l_20pct_null_images_add_rawData_batch_4_final.pt",
    # "yolo11s_20pct_null_images_add_rawData_batch_4_final.pt",
    # "yolo11n_20pct_null_images_add_rawData_batch_4_final.pt",
    # "yolo11x_20pct_null_images_add_rawData_batch_4_final.pt",
    # "yolo11s_20pct_null_images_add_rawData_batch_16_final.pt",
    # "yolo11n_20pct_null_images_add_rawData_batch_16_final.pt",
    # "yolo11n_20pct_null_images_add_rawData_batch_8_final.pt",
    # "yolo11l_20pct_null_images_add_rawData_batch_16_final.pt",
    # "yolo11l_20pct_null_images_add_rawData_batch_8_final.pt",
    # "yolo11s_20pct_null_images_add_rawData_batch_8_final.pt",
    # "yolo11m_20pct_null_images_add_rawData_batch_4_final.pt"
]

CONF = 0.01
DEVICE = "cuda"

SLICE = 640
OVERLAP = 0.2

SKIP_IF_PRED_EXISTS = True


# =========================
# Helpers
# =========================
def get_subdirs():
    return [p for p in DATA_ROOT.iterdir() if p.is_dir()]


def model_tag(p):
    return Path(p).stem


def ensure_dataset(site_dir):
    name = f"{VERSION}_{site_dir.name}"

    if name in fo.list_datasets():
        return fo.load_dataset(name)

    img_dir = site_dir

    ds = fo.Dataset.from_images_dir(str(img_dir), name=name)
    return ds


def ensure_field(ds, field):
    if field not in ds.get_field_schema():
        ds.add_sample_field(
            field,
            fo.EmbeddedDocumentField,
            embedded_doc_type=fo.Detections,
        )


def run_model(ds, ckpt_path, pred_field):

    ensure_field(ds, pred_field)

    model = AutoDetectionModel.from_pretrained(
        model_type="yolov8",
        model_path=str(ckpt_path),
        confidence_threshold=CONF,
        image_size=640,
        device=DEVICE,
    )

    for sample in ds.iter_samples(progress=True, autosave=True):

        res = get_sliced_prediction(
            sample.filepath,
            model,
            slice_height=SLICE,
            slice_width=SLICE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            verbose=0,
        )

        sample[pred_field] = fo.Detections(
            detections=res.to_fiftyone_detections()
        )


# =========================
# RUN
# =========================
site_dirs = get_subdirs()
display(site_dirs)
# site_dirs = [site_dirs[-1]]  # 测试只跑最后一个子目录
# print(f"Processing site dir: {site_dirs}")

[PosixPath('/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data/sw1'),
 PosixPath('/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data/lloyd'),
 PosixPath('/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data/air1'),
 PosixPath('/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data/southfarm1'),
 PosixPath('/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data/ms1'),
 PosixPath('/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data/sw2'),
 PosixPath('/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a05_2024_data/jeff')]

In [3]:

for site in site_dirs:

    ds = ensure_dataset(site)

    print("\n==== DATASET ====", ds.name)

    for m in MODEL_NAMES:

        ckpt = MODEL_DIR / m
        tag = model_tag(ckpt)

        pred_field = f"pred_{tag}"

        if SKIP_IF_PRED_EXISTS and pred_field in ds.get_field_schema():
            if ds.count(f"{pred_field}.detections") > 0:
                print("[SKIP]", tag)
                continue

        print("[RUN]", tag)
        run_model(ds, ckpt, pred_field)

print("\nDONE")


 100% |███████████████| 9081/9081 [861.0ms elapsed, 0s remaining, 10.6K samples/s]      

==== DATASET ==== sahi_null_run_whole_rawData_rest7datasets_v7_sw1
[RUN] yolo11l_20pct_null_images_add_rawData_batch_4_final
 100% |███████████████| 9081/9081 [3.2h elapsed, 0s remaining, 0.7 samples/s]    
 100% |███████████████| 6859/6859 [604.8ms elapsed, 0s remaining, 11.4K samples/s]      

==== DATASET ==== sahi_null_run_whole_rawData_rest7datasets_v7_lloyd
[RUN] yolo11l_20pct_null_images_add_rawData_batch_4_final
 100% |███████████████| 6859/6859 [2.6h elapsed, 0s remaining, 0.7 samples/s]    
 100% |█████████████████| 348/348 [41.3ms elapsed, 0s remaining, 8.4K samples/s]   

==== DATASET ==== sahi_null_run_whole_rawData_rest7datasets_v7_air1
[RUN] yolo11l_20pct_null_images_add_rawData_batch_4_final
 100% |█████████████████| 348/348 [7.5m elapsed, 0s remaining, 0.8 samples/s]    
 100% |█████████████████| 240/240 [27.4ms elapsed, 0s remaining, 8.7K samples/s]   

==== DATASET ==== sahi_nul

### 导出两个 Excel（整体 + 每图）

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import List, Dict, Any
from datetime import datetime

import numpy as np
import pandas as pd
import fiftyone as fo


# =========================
# Config
# =========================
VERSION = "sahi_null_run_rawData_v4"
OUT_DIR = Path("./_pred_exports2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 你预测时用的模型名（用来拼 pred_field）
MODEL_NAMES = [
    "yolo11n_20pct_null_images_add_rawData_batch_8_final.pt",
    # "yolo11x_20pct_null_images_add_rawData_batch_8_final.pt",
    # "yolo11m_20pct_null_images_add_rawData_batch_16_final.pt",
]
def model_tag_from_name(name: str) -> str:
    return Path(name).stem  # 你 pred_field 用 pred_{stem}

PRED_FIELDS = {model_tag_from_name(n): f"pred_{model_tag_from_name(n)}" for n in MODEL_NAMES}

# 输出文件名
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_SUMMARY_XLSX = OUT_DIR / f"pred_summary__{VERSION}__{stamp}.xlsx"
OUT_PER_IMAGE_XLSX = OUT_DIR / f"pred_per_image__{VERSION}__{stamp}.xlsx"


# =========================
# Helpers
# =========================
def safe_stats(confs: List[float]) -> Dict[str, float]:
    if not confs:
        return {"conf_max": np.nan, "conf_mean": np.nan, "conf_p50": np.nan, "conf_p90": np.nan}
    arr = np.asarray(confs, dtype=float)
    return {
        "conf_max": float(np.nanmax(arr)),
        "conf_mean": float(np.nanmean(arr)),
        "conf_p50": float(np.nanpercentile(arr, 50)),
        "conf_p90": float(np.nanpercentile(arr, 90)),
    }


def get_detections(sample: fo.Sample, pred_field: str):
    """
    安全拿 detections list
    """
    if not sample.has_field(pred_field):
        return []
    val = sample[pred_field]  # 可能是 None / Detections
    if val is None:
        return []
    dets = getattr(val, "detections", None)
    return dets or []


def export_two_excels_for_dataset(ds: fo.Dataset, pred_fields: Dict[str, str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    ds_name = ds.name
    n_samples = len(ds)

    summary_rows: List[Dict[str, Any]] = []
    per_image_rows: List[Dict[str, Any]] = []

    for model_tag, pred_field in pred_fields.items():
        if pred_field not in ds.get_field_schema():
            print(f"[WARN] {ds_name} missing pred_field={pred_field}, skip")
            continue

        total_pred = int(ds.count(f"{pred_field}.detections"))

        images_with_pred = 0
        confs_all: List[float] = []

        for sample in ds.iter_samples(progress=True):
            det_list = get_detections(sample, pred_field)
            n_det = len(det_list)

            if n_det > 0:
                images_with_pred += 1

            confs = [float(d.confidence) for d in det_list if d.confidence is not None]
            confs_all.extend(confs)

            row = {
                "dataset": ds_name,
                "sample_id": str(sample.id),
                "filepath": sample.filepath,
                "filename": Path(sample.filepath).name,
                "model_tag": model_tag,
                "pred_field": pred_field,
                "pred_count": n_det,
            }
            row.update(safe_stats(confs))
            per_image_rows.append(row)

        s = {
            "dataset": ds_name,
            "model_tag": model_tag,
            "pred_field": pred_field,
            "num_samples": n_samples,
            "total_pred_count": total_pred,
            "images_with_pred": images_with_pred,
            "pct_images_with_pred": (images_with_pred / n_samples * 100.0) if n_samples else np.nan,
            "avg_pred_per_image": (total_pred / n_samples) if n_samples else np.nan,
        }
        s.update(safe_stats(confs_all))
        summary_rows.append(s)

    return pd.DataFrame(summary_rows), pd.DataFrame(per_image_rows)


# =========================
# RUN
# =========================
all_summary = []
all_per_image = []

ds_names = sorted([n for n in fo.list_datasets() if n.startswith(f"{VERSION}_")])
print("Found datasets:", len(ds_names))
for n in ds_names:
    print(" -", n)

for ds_name in ds_names:
    ds = fo.load_dataset(ds_name)
    print(f"\n[EXPORT] {ds_name} | samples={len(ds)}")
    df_sum, df_img = export_two_excels_for_dataset(ds, PRED_FIELDS)
    all_summary.append(df_sum)
    all_per_image.append(df_img)

df_summary = pd.concat(all_summary, ignore_index=True) if all_summary else pd.DataFrame()
df_per_image = pd.concat(all_per_image, ignore_index=True) if all_per_image else pd.DataFrame()

if not df_summary.empty:
    df_summary = df_summary.sort_values(["dataset", "model_tag"]).reset_index(drop=True)
if not df_per_image.empty:
    df_per_image = df_per_image.sort_values(["dataset", "model_tag", "filename"]).reset_index(drop=True)

with pd.ExcelWriter(OUT_SUMMARY_XLSX, engine="openpyxl") as w:
    df_summary.to_excel(w, sheet_name="summary", index=False)

with pd.ExcelWriter(OUT_PER_IMAGE_XLSX, engine="openpyxl") as w:
    df_per_image.to_excel(w, sheet_name="per_image", index=False)

print("\nSaved:")
print(" -", OUT_SUMMARY_XLSX)
print(" -", OUT_PER_IMAGE_XLSX)
print("summary shape:", df_summary.shape, "| per_image shape:", df_per_image.shape)


Found datasets: 14
 - sahi_null_run_rawData_v4_jeff_0613-0624_04_ok
 - sahi_null_run_rawData_v4_jeff_0624-0702_01_ok
 - sahi_null_run_rawData_v4_jeff_0730-0813_01
 - sahi_null_run_rawData_v4_ms1_0605-0621_40
 - sahi_null_run_rawData_v4_ms1_0621-0710_04
 - sahi_null_run_rawData_v4_ms1_0710-0726_36
 - sahi_null_run_rawData_v4_ms1_0726-0809_11
 - sahi_null_run_rawData_v4_ms1_0809-0823_34
 - sahi_null_run_rawData_v4_ms2_0605-0621_30
 - sahi_null_run_rawData_v4_ms2_0621-0710_01
 - sahi_null_run_rawData_v4_ms2_0710-0726_41
 - sahi_null_run_rawData_v4_ms2_0726-0809_13
 - sahi_null_run_rawData_v4_ms2_0809-0823_10
 - sahi_null_run_rawData_v4_ms2_0823-0906_07

[EXPORT] sahi_null_run_rawData_v4_jeff_0613-0624_04_ok | samples=1112
 100% |███████████████| 1112/1112 [146.2ms elapsed, 0s remaining, 7.6K samples/s]     

[EXPORT] sahi_null_run_rawData_v4_jeff_0624-0702_01_ok | samples=768
 100% |█████████████████| 768/768 [63.7ms elapsed, 0s remaining, 12.1K samples/s] 

[EXPORT] sahi_null_run_rawData